# Modern Coding Practice — Log Aggregator (Amazon FAR style)

One domain, escalating constraints. Parts 1-3 build the core (concurrency at Part 3); Parts 4-5 are
stretch tasks (out-of-order/late + malformed tolerance, then a parallel map-reduce). Fill stubs, run
each test cell, peek at the collapsed `ref_` solutions only after trying.

Log line format: `"<epoch_int> <LEVEL> <message>"`, `LEVEL in {DEBUG, INFO, WARN, ERROR}`.

---

## Part 1 — Parse a log line

`parse_line(line) -> (ts: int, level: str, message: str)`. Raise `ValueError` on anything malformed:
non-integer timestamp, unknown level, or a missing message.

**Lock down:** message can contain spaces (split only twice). Is an empty message valid? (No.) Are
negative timestamps allowed? (Yes — clock can be pre-epoch in tests.)

In [8]:
LEVELS = {"DEBUG", "INFO", "WARN", "ERROR"}


def parse_line(line):
    try:
        chunks = line.split(" ")
        ts = int(chunks[0])
        lvl = chunks[1]
        if lvl not in LEVELS:
            raise ValueError(f"level not valid: {lvl}")
        msg = ' '.join(chunks[2:])
        if len(msg) == 0:
            raise ValueError("empty message")
        return (ts, lvl, msg)
    except:
        raise ValueError()

In [4]:
# --- Part 1 tests ---
def _t1():
    assert parse_line("100 INFO started") == (100, "INFO", "started")
    assert parse_line("5 ERROR disk full now") == (5, "ERROR", "disk full now")
    for bad in ("", "abc INFO x", "100 NOPE x", "100 INFO"):
        try:
            parse_line(bad)
        except ValueError:
            pass
        else:
            raise AssertionError("expected ValueError for %r" % bad)
    print("Part 1 OK")

_t1()

Part 1 OK


## Part 2 — K-way merge of sorted streams

`merge_sorted(streams) -> list` merges K already-sorted record streams into one list ordered by
timestamp. On ties, preserve stream order (stable).

**Lock down:** streams may be huge/lazy — prefer a heap (`heapq.merge`) over concatenate-then-sort
so it is O(N log K) and streaming-friendly. Why stability matters for tie-break determinism.

In [19]:
import heapq


def merge_sorted(streams):
    h = []
    for i, stream in enumerate(streams):
        heapq.heappush(h, (stream[0][0], i, 0, stream[0]))
    res = []
    while h:
        ts, stream_idx, idx, item = heapq.heappop(h)
        res.append(item)
        if len(streams[stream_idx]) > idx + 1:
            heapq.heappush(h, (streams[stream_idx][idx + 1][0], stream_idx, idx + 1, streams[stream_idx][idx + 1]))
    return res

In [20]:
# --- Part 2 tests ---
def _t2():
    a = [(1, "INFO", "a1"), (4, "INFO", "a4")]
    b = [(2, "WARN", "b2"), (3, "WARN", "b3"), (5, "WARN", "b5")]
    out = merge_sorted([a, b])
    assert [r[0] for r in out] == [1, 2, 3, 4, 5]
    c = [(2, "INFO", "c2")]
    tie = merge_sorted([b, c])               # both have ts=2; b's stream comes first
    assert tie[0] == (2, "WARN", "b2") and tie[1] == (2, "INFO", "c2")
    print("Part 2 OK")

_t2()

Part 2 OK


## Part 3 — Concurrent multi-source ingest

`ingest(sources) -> list`: each source is a list of raw lines. Parse each source in its own thread,
push records onto a shared `queue.Queue`, and a consumer drains until every producer has signalled
done, returning all records sorted by timestamp.

**Discuss:** sentinel-per-producer vs `Queue.join`/`task_done`; why a final sort is needed (arrival
order across threads is non-deterministic); bounded queue = backpressure on fast producers; this is
I/O/coordination work so threads are appropriate (Part 5 goes to processes for CPU-bound counting).

In [27]:
import queue
import threading
from concurrent.futures import ThreadPoolExecutor

q = queue.Queue()
def parse_line(line):
    try:
        chunks = line.split(" ")
        ts = int(chunks[0])
        lvl = chunks[1]
        if lvl not in LEVELS:
            raise ValueError(f"level not valid: {lvl}")
        msg = ' '.join(chunks[2:])
        if len(msg) == 0:
            raise ValueError("empty message")
        return (ts, lvl, msg)
    except:
        raise ValueError()
        
def producer(source, event):
    for line in source:
        print(line)
        res = parse_line(line)
        q.put(res)
    event.set()

def consumer(events):
    print("consumer started")
    out = []
    while not all(e.is_set() for e in events) or not q.empty():
        r = q.get(timeout=3)
        print(r)
        out.append(r)
    out.sort()
    return out
        
        
    
def ingest(sources):
    events = []
    tasks = []

    with ThreadPoolExecutor() as e:
        for source in sources:
            event = threading.Event()
            task = e.submit(producer, source, event)
            
            events.append(event)
            tasks.append(task)
        
        consumer_task = e.submit(consumer, events)
    for i, task in enumerate(tasks):
        task.result(timeout=3)
        events[i].set()
    res = consumer_task.result(timeout=5)
    print(res)
    return res

In [28]:
# --- Part 3 tests ---
def _t3():
    s1 = ["1 INFO a", "3 INFO b"]
    s2 = ["2 WARN c", "4 ERROR d"]
    out = ingest([s1, s2])
    assert [r[0] for r in out] == [1, 2, 3, 4]   # sorted regardless of thread timing
    assert len(out) == 4
    print("Part 3 OK")

_t3()

1 INFO a2 WARN c
4 ERROR d

3 INFO b
consumer started
(2, 'WARN', 'c')
(4, 'ERROR', 'd')
(1, 'INFO', 'a')
(3, 'INFO', 'b')
[(1, 'INFO', 'a'), (2, 'WARN', 'c'), (3, 'INFO', 'b'), (4, 'ERROR', 'd')]
Part 3 OK


## Part 4 (stretch) — Out-of-order, late & malformed events

A real stream arrives roughly-but-not-exactly in order. Build `StreamAggregator(max_lateness)`:

- `push_line(line) -> list`: parse and buffer; **return records that are now safe to emit**, in
  timestamp order. "Safe" = `ts <= watermark`, where `watermark = max_ts_seen - max_lateness`.
- Skip malformed lines (count them in `dropped_malformed`, emit nothing).
- Drop events older than the watermark as too-late (count in `dropped_late`).
- `flush() -> list`: emit everything still buffered, in order.

**Discuss:** event-time vs processing-time, watermarks, the latency/completeness trade-off in
`max_lateness`, and what you do with dropped-late events (dead-letter queue).

In [ ]:
import heapq


class StreamAggregator:
    def __init__(self, max_lateness):
        raise NotImplementedError

    def push_line(self, line):
        raise NotImplementedError

    def flush(self):
        raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    agg = StreamAggregator(max_lateness=5)
    assert agg.push_line("10 INFO a") == []          # watermark 5, nothing safe yet
    assert agg.push_line("11 INFO b") == []
    out = agg.push_line("20 INFO c")                 # watermark 15 -> release 10, 11
    assert [r[0] for r in out] == [10, 11]
    assert agg.push_line("bad line") == [] and agg.dropped_malformed == 1
    assert agg.push_line("3 INFO late") == [] and agg.dropped_late == 1   # 3 < watermark 15
    assert [r[0] for r in agg.flush()] == [20]
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Parallel map-reduce over many files

`parallel_count(files, bucket_seconds) -> dict`: count records by level and by fixed-size time
bucket across many files (CPU-bound), using `ProcessPoolExecutor`. Each file is a list of lines; the
worker is `logagg_workers.count_file`. Merge partials in the parent.

Return `{"levels": {level: n}, "buckets": {bucket_start: n}}`; malformed lines are skipped.

**Discuss:** map-reduce shape (per-file map, combine in parent), GIL (counting is CPU-bound ->
processes), shipping lines vs file paths to reduce pickling, and combiner associativity.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from collections import Counter
import logagg_workers


def parallel_count(files, bucket_seconds):
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    f1 = ["0 INFO a", "5 INFO b", "65 ERROR c", "garbage"]
    f2 = ["10 WARN d", "70 INFO e"]
    res = parallel_count([f1, f2], bucket_seconds=60)
    assert res["levels"] == {"INFO": 3, "ERROR": 1, "WARN": 1}, res
    assert res["buckets"] == {0: 3, 60: 2}, res     # 0/5/10 -> bucket 0; 65/70 -> bucket 60
    print("Part 5 OK")

_t5()

## Likely follow-ups
- Exactly-once vs at-least-once delivery; dedupe by event id.
- Backpressure end-to-end; spilling the reorder buffer to disk when lateness is large.
- Streaming windows (tumbling/sliding/session) and incremental aggregation.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import heapq
import queue
import threading
from collections import Counter
from concurrent.futures import ProcessPoolExecutor
import logagg_workers

_LEVELS = {"DEBUG", "INFO", "WARN", "ERROR"}


def ref_parse_line(line):
    parts = line.split(" ", 2)
    if len(parts) < 3:
        raise ValueError("malformed: need ts level message")
    ts_s, level, msg = parts
    if not ts_s.lstrip("-").isdigit():
        raise ValueError("bad timestamp")
    if level not in _LEVELS:
        raise ValueError("bad level")
    return (int(ts_s), level, msg)


def ref_merge(streams):
    return list(heapq.merge(*streams, key=lambda r: r[0]))   # stable across streams


def ref_ingest(sources):
    q = queue.Queue(maxsize=1000)
    sentinel = object()

    def producer(lines):
        for ln in lines:
            q.put(ref_parse_line(ln))
        q.put(sentinel)

    threads = [threading.Thread(target=producer, args=(s,)) for s in sources]
    for t in threads:
        t.start()
    out, done = [], 0
    while done < len(sources):
        item = q.get()
        if item is sentinel:
            done += 1
        else:
            out.append(item)
    for t in threads:
        t.join()
    out.sort(key=lambda r: r[0])
    return out


class RefStreamAgg:
    def __init__(self, max_lateness):
        self.max_lateness = max_lateness
        self.buf = []                 # min-heap by (ts, seq)
        self.max_ts = None
        self.dropped_malformed = 0
        self.dropped_late = 0
        self._seq = 0

    def push_line(self, line):
        try:
            rec = ref_parse_line(line)
        except ValueError:
            self.dropped_malformed += 1
            return []
        ts = rec[0]
        if self.max_ts is None or ts > self.max_ts:
            self.max_ts = ts
        watermark = self.max_ts - self.max_lateness
        if ts < watermark:
            self.dropped_late += 1
            return []
        heapq.heappush(self.buf, (ts, self._seq, rec))
        self._seq += 1
        out = []
        while self.buf and self.buf[0][0] <= watermark:
            out.append(heapq.heappop(self.buf)[2])
        return out

    def flush(self):
        out = []
        while self.buf:
            out.append(heapq.heappop(self.buf)[2])
        return out


def ref_parallel_count(files, bucket_seconds):
    items = [(f, bucket_seconds) for f in files]
    levels, buckets = Counter(), Counter()
    with ProcessPoolExecutor() as ex:
        for lv, bk in ex.map(logagg_workers.count_file, items):
            levels.update(lv)
            buckets.update(bk)
    return {"levels": dict(levels), "buckets": dict(buckets)}


assert ref_parse_line("9 WARN hi there") == (9, "WARN", "hi there")
assert [r[0] for r in ref_merge([[(1, "INFO", "x")], [(0, "INFO", "y"), (2, "INFO", "z")]])] == [0, 1, 2]
_a = RefStreamAgg(5)
assert _a.push_line("10 INFO a") == [] and [r[0] for r in _a.push_line("20 INFO c")] == [10]
assert ref_parallel_count([["0 INFO a", "65 ERROR c"]], 60) == {"levels": {"INFO": 1, "ERROR": 1}, "buckets": {0: 1, 60: 1}}
print("reference solutions OK")